# 人体姿勢推定＋3ディアボロのオブジェクト追跡

`pose_video_materials.ipynb` のコピーに、参考元 `diabolo_tracking/dev.ipynb` の **DaSiamRPN** 追跡を追加しました。
人体はYOLO Pose、ディアボロは手動指定した領域をDaSiamRPNで追跡します。自動物体検出ではありません。

1. プロジェクト直下で `uv sync` → `uv run jupyter lab pose_object_tracking.ipynb`。
2. 設定を確認して上から実行。開始フレーム上のディアボロを **3つ順に** 囲み、各選択をEnter/Spaceで確定します（Escは中止）。D1〜D3は選択順です。
3. `tracking.mp4`（追跡枠のみ）と `pose_tracking.mp4`（骨格＋追跡枠）、`skeleton_tracking.mp4`（黒背景の骨格＋追跡枠）、元の3素材、`tracking.csv`、`manifest.json` を確認します。動画はすべて無音・同一FPS・同一フレーム数です。

初期設定は現在の動画・YOLO26x Poseを引き継ぎ、試行区間を先頭5秒にしています。FFmpegが必要です。
3つの追跡器は独立しており、交差・遮蔽で同じ物体を追ったりIDが入れ替わる場合があります。スコアは正解確率ではありません。低スコアや失敗時は枠を非表示にし、CSVに記録します。補間や自動再検出はしません。
スティック・紐、手首のひねり・指・奥行きは推定しません。推定を正しい動作と断定せず、元映像と比較してください。

モデルはこのプロジェクトの `models/` に配置します。API参考: [OpenCV DaSiamRPN](https://docs.opencv.org/4.13.0/de/d93/classcv_1_1TrackerDaSiamRPN.html)。


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import csv
import math
import shutil
import subprocess
import uuid

import cv2
import numpy as np
import ultralytics
from ultralytics import YOLO
from IPython.display import Video, display

ROOT = Path.cwd()
if not (ROOT / "assets").is_dir():
    raise RuntimeError("プロジェクト直下を作業ディレクトリとして起動してください。")

print("Ultralytics:", ultralytics.__version__)
print("入力動画候補:")
for path in sorted((ROOT / "assets").iterdir()):
    print(" ", path.relative_to(ROOT))


## 1. 設定

`INPUT_VIDEO` に処理したいファイルを指定します。初期値は動作確認用の仮選択です。

`MODEL` だけ変更して別モデルを試せます。例：`str(ROOT / "models" / "yolo11m-pose.pt")`、YOLO26対応環境では `str(ROOT / "models" / "yolo26m-pose.pt")`。ファイルが手元にないモデルはUltralyticsがダウンロードします。

`CROP_XYWH` は回転補正後の画面に対する `(左端, 上端, 幅, 高さ)`。3素材に同じ切り抜きを適用します。動画の回転メタデータはOpenCVの自動処理に任せ、ここで90度回転を強制しません。


In [ ]:
INPUT_VIDEO = ROOT / "assets" / "ヤマザキ.mp4"
MODEL = str(ROOT / "models" / "yolo26x-pose.pt")

START_SECONDS = 0.5
DURATION_SECONDS = 30
CROP_XYWH = None
# CROP_XYWH = (400, 0, 600, 720)      # アゴウテンセイ
# CROP_XYWH = (380, 0, 600, 720)      # トミタヒカル
OUTPUT_WIDTH = 640     # 各素材の幅。縦横比を維持し、偶数サイズに調整
PERSON_CONFIDENCE = 0.25
KEYPOINT_CONFIDENCE = 0.5
IMAGE_SIZE = 640
DEVICE = "mps"        # Apple Silicon: "mps"、CUDA: 0
OUTPUT_ROOT = ROOT / "outputs"

# 対象数は3つ固定。Noneなら開始フレームで順に囲む。
# 手入力の場合は [(x, y, w, h), (...), (...)] の3組。
# 座標は回転補正・CROP適用後、縮小前のフレーム基準。
TRACKING_ROIS = None
TRACKING_MIN_SCORE = 0.5  # 表示の目安。正解確率ではない
TRACKING_MODELS = {
    "model": ROOT / "models" / "dasiamrpn_model.onnx",
    "kernel_r1": ROOT / "models" / "dasiamrpn_kernel_r1.onnx",
    "kernel_cls1": ROOT / "models" / "dasiamrpn_kernel_cls1.onnx",
}
TRACKING_COLORS = [(80, 220, 80), (220, 80, 220), (40, 220, 240)]


## 2. 骨格の描画

演者の身体の左側はオレンジ、右側は水色、中央はグレーで描画します。画面上の左右ではありません。
関節の信頼度が閾値未満なら点を描かず、両端が有効な線だけを描きます。人物未検出でもフレームを省略せず、骨格素材には黒背景を出力します。ちらつきを隠す補間・平滑化は行いません。


In [ ]:
# COCO人体17点（0始まり）。色はOpenCVのBGR順。
LEFT = {1, 3, 5, 7, 9, 11, 13, 15}
RIGHT = {2, 4, 6, 8, 10, 12, 14, 16}
LEFT_COLOR = (40, 160, 255)
RIGHT_COLOR = (255, 210, 60)
CENTER_COLOR = (210, 210, 210)
EDGES = [(0, 1), (0, 2), (1, 3), (2, 4), (5, 6),
         (5, 7), (7, 9), (6, 8), (8, 10), (5, 11), (6, 12),
         (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)]


def draw_pose(canvas, keypoints, threshold):
    """同じ座標と信頼度から、背景によらず同じ骨格を描画する。"""
    height, width = canvas.shape[:2]
    for person in keypoints:
        xy = person[:, :2]
        valid = (np.isfinite(person).all(axis=1)
                 & (person[:, 2] >= threshold)
                 & (xy[:, 0] >= 0) & (xy[:, 0] < width)
                 & (xy[:, 1] >= 0) & (xy[:, 1] < height))
        for a, b in EDGES:
            if valid[a] and valid[b]:
                color = LEFT_COLOR if a in LEFT and b in LEFT else (
                    RIGHT_COLOR if a in RIGHT and b in RIGHT else CENTER_COLOR)
                cv2.line(canvas, tuple(xy[a].astype(int)), tuple(xy[b].astype(int)),
                         color, 2, cv2.LINE_AA)
        for index in np.flatnonzero(valid):
            color = LEFT_COLOR if index in LEFT else (
                RIGHT_COLOR if index in RIGHT else CENTER_COLOR)
            cv2.circle(canvas, tuple(xy[index].astype(int)), 4, color, -1, cv2.LINE_AA)
    return canvas


## 3. 同期した素材を生成

入力は順番に読み、選択区間だけを推定・保存します。フレームを動画全体ぶんメモリに蓄積しません。1回の推論結果を骨格のみ・重ね合わせの両方に使います。

出力は毎回別フォルダに保存され、既存素材は上書きしません。途中で停止・失敗した場合は未完了のファイルが残る場合があります。`manifest.json` と最後の「完了」が成功の目印です。

入力の公称FPSで固定フレームレートの素材を作成します。可変フレームレートの入力では、入力音声との厳密な時刻一致までは保証しません。音声同期が必要な場合は入力を事前に固定FPSへ変換してください。


In [ ]:
if not INPUT_VIDEO.is_file():
    raise FileNotFoundError(INPUT_VIDEO)
if not shutil.which("ffmpeg"):
    raise RuntimeError("FFmpegが必要です。インストールしてから実行してください。")
if not math.isfinite(START_SECONDS) or START_SECONDS < 0:
    raise ValueError("START_SECONDS は0以上の有限値にしてください。")
if DURATION_SECONDS is not None and (
        not math.isfinite(DURATION_SECONDS) or DURATION_SECONDS <= 0):
    raise ValueError("DURATION_SECONDS は正の有限値またはNoneにしてください。")
if not isinstance(OUTPUT_WIDTH, int) or OUTPUT_WIDTH < 2:
    raise ValueError("OUTPUT_WIDTH は2以上の整数にしてください。")
if not (0 <= PERSON_CONFIDENCE <= 1 and 0 <= KEYPOINT_CONFIDENCE <= 1):
    raise ValueError("信頼度の閾値は0〜1にしてください。")

if not 0 <= TRACKING_MIN_SCORE <= 1:
    raise ValueError("TRACKING_MIN_SCORE は0〜1にしてください。")
for path in TRACKING_MODELS.values():
    if not path.is_file():
        raise FileNotFoundError(f"追跡モデルが必要です: {path}")
params = cv2.TrackerDaSiamRPN_Params()
for key, path in TRACKING_MODELS.items():
    setattr(params, key, str(path))
params.backend = cv2.dnn.DNN_BACKEND_OPENCV
params.target = cv2.dnn.DNN_TARGET_CPU
trackers = []
selected_rois = []
tracking_rows = []
model = YOLO(MODEL)
if model.task != "pose":
    raise ValueError("-pose.pt の人体姿勢推定モデルを指定してください。")

run_name = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
OUTPUT_DIR = OUTPUT_ROOT / (INPUT_VIDEO.stem + "_" + run_name)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
NAMES = ("skeleton", "original", "overlay", "tracking", "pose_tracking", "skeleton_tracking")
raw_paths = {name: OUTPUT_DIR / f"{name}.working.mp4" for name in NAMES}
final_paths = {name: OUTPUT_DIR / f"{name}.mp4" for name in NAMES}
cap = cv2.VideoCapture(str(INPUT_VIDEO))
writers = {}
written = 0
missing = 0
try:
    if not cap.isOpened():
        raise RuntimeError("入力動画を開けません。")
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not math.isfinite(fps) or fps <= 0:
        raise ValueError("入力動画のFPSを取得できません。")
    start_frame = round(START_SECONDS * fps)
    limit = None if DURATION_SECONDS is None else max(1, round(DURATION_SECONDS * fps))
    frame_index = 0
    while limit is None or written < limit:
        ok, frame = cap.read()
        if not ok:
            break
        current_index = frame_index
        frame_index += 1
        if current_index < start_frame:
            continue
        if CROP_XYWH is not None:
            if len(CROP_XYWH) != 4 or not all(isinstance(v, int) for v in CROP_XYWH):
                raise ValueError("CROP_XYWH は4つの整数で指定してください。")
            x, y, w, h = CROP_XYWH
            height, width = frame.shape[:2]
            if x < 0 or y < 0 or w <= 0 or h <= 0 or x + w > width or y + h > height:
                raise ValueError(f"切り抜き範囲が動画の外に出ています。 height: {height}, width: {width}")
            frame = frame[y:y+h, x:x+w]
        if not writers:
            source_shape = frame.shape[:2]
            output_w = OUTPUT_WIDTH // 2 * 2
            output_h = max(2, round(source_shape[0] * output_w / source_shape[1] / 2) * 2)
            frame_size = (output_w, output_h)
            for name in NAMES:
                writer = cv2.VideoWriter(str(raw_paths[name]), cv2.VideoWriter_fourcc(*"mp4v"),
                                         fps, frame_size)
                writers[name] = writer
                if not writer.isOpened():
                    raise RuntimeError(f"動画を書き込めません: {raw_paths[name]}")
        if frame.shape[:2] != source_shape:
            raise ValueError("入力動画の途中で解像度が変わりました。")
        # 最初の対象フレームで3つを指定。描画前のフレームを追跡器へ渡す。
        initialized_now = not trackers
        if initialized_now:
            rois = TRACKING_ROIS
            if rois is None:
                rois = []
                preview = frame.copy()
                try:
                    for object_id, color in enumerate(TRACKING_COLORS, start=1):
                        title = f"Select D{object_id} / 3 - Enter: accept, Esc: cancel"
                        roi = tuple(map(int, cv2.selectROI(title, preview, False, True)))
                        cv2.destroyWindow(title)
                        if roi[2] <= 0 or roi[3] <= 0:
                            raise RuntimeError("領域選択を中止しました。セルを再実行してください。")
                        rois.append(roi)
                        x, y, w, h = roi
                        cv2.rectangle(preview, (x, y), (x+w, y+h), color, 2)
                finally:
                    cv2.destroyAllWindows()
            if len(rois) != 3:
                raise ValueError("TRACKING_ROIS は3つの領域を指定してください。")
            for roi in rois:
                if len(roi) != 4 or not all(isinstance(v, int) for v in roi):
                    raise ValueError("各領域は整数の (x, y, w, h) にしてください。")
                x, y, w, h = roi
                if x < 0 or y < 0 or w <= 0 or h <= 0 or x+w > frame.shape[1] or y+h > frame.shape[0]:
                    raise ValueError("追跡領域がフレーム外です。")
                tracker = cv2.TrackerDaSiamRPN_create(params)
                tracker.init(frame, tuple(roi))
                trackers.append(tracker)
                selected_rois.append(tuple(roi))
        result = model.predict(frame, imgsz=IMAGE_SIZE, conf=PERSON_CONFIDENCE,
                               device=DEVICE, verbose=False)[0]
        points = np.empty((0, 17, 3), dtype=np.float32)
        if result.keypoints is not None and len(result.keypoints):
            points = result.keypoints.data.cpu().numpy().copy()
            if points.shape[1:] != (17, 3):
                raise ValueError("COCO人体17点と信頼度を返すモデルが必要です。")
            # 検出枠の面積が最大の人物だけ表示。同面積なら先頭を採用。
            boxes = result.boxes.xyxy.cpu().numpy()
            areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
            largest = int(np.argmax(areas))
            points = points[largest:largest + 1]
            points[:, :, 0] *= output_w / frame.shape[1]
            points[:, :, 1] *= output_h / frame.shape[0]
        else:
            missing += 1
        original = cv2.resize(frame, frame_size, interpolation=cv2.INTER_AREA)
        overlay = draw_pose(original.copy(), points, KEYPOINT_CONFIDENCE)
        skeleton = draw_pose(np.zeros_like(original), points, KEYPOINT_CONFIDENCE)
        tracking = original.copy()
        pose_tracking = overlay.copy()
        skeleton_tracking = skeleton.copy()
        for object_id, (tracker, color) in enumerate(zip(trackers, TRACKING_COLORS), start=1):
            if initialized_now:
                success, bbox, score = True, selected_rois[object_id-1], None
                status = "initialized"
            else:
                success, bbox = tracker.update(frame)
                score = float(tracker.getTrackingScore())
                status = "tracked" if success and math.isfinite(score) and score >= TRACKING_MIN_SCORE else "uncertain"
            valid_box = (len(bbox) == 4 and np.isfinite(bbox).all()
                         and bbox[2] > 0 and bbox[3] > 0
                         and bbox[0] >= 0 and bbox[1] >= 0
                         and bbox[0]+bbox[2] <= frame.shape[1]
                         and bbox[1]+bbox[3] <= frame.shape[0])
            accepted = valid_box and status in ("initialized", "tracked")
            if not valid_box:
                status = "invalid_box"
            # CSVは切り抜き後・縮小前の座標。採用できない座標は空欄。
            values = tuple(map(float, bbox)) if accepted else (None,)*4
            tracking_rows.append({
                "frame": written, "source_frame": current_index,
                "time_seconds": current_index / fps, "object_id": object_id,
                "status": status, "update_ok": bool(success),
                "score": score if score is not None and math.isfinite(score) else None,
                **dict(zip(("x", "y", "w", "h"), values)),
            })
            for canvas in (tracking, pose_tracking, skeleton_tracking):
                if accepted:
                    x, y, w, h = bbox
                    a = (round(x * output_w / frame.shape[1]), round(y * output_h / frame.shape[0]))
                    b = (round((x+w) * output_w / frame.shape[1]), round((y+h) * output_h / frame.shape[0]))
                    cv2.rectangle(canvas, a, b, color, 2)
                    cv2.putText(canvas, f"D{object_id}", (a[0], max(16, a[1]-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                else:
                    cv2.putText(canvas, f"D{object_id}: uncertain", (8, 24*object_id), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        for name, image in (("skeleton", skeleton), ("original", original), ("overlay", overlay),
                            ("tracking", tracking), ("pose_tracking", pose_tracking),
                            ("skeleton_tracking", skeleton_tracking)):
            writers[name].write(image)
        written += 1
        if written % 100 == 0:
            print(f"{written}フレーム処理済み")
finally:
    cap.release()
    for writer in writers.values():
        writer.release()

if written == 0:
    raise RuntimeError("処理対象のフレームがありません。入力と開始時刻を確認してください。")

# 編集ソフト・ブラウザで使いやすいH.264/yuv420pへ変換。
for name in NAMES:
    process = subprocess.run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-n", "-i", str(raw_paths[name]),
        "-an", "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p",
        "-movflags", "+faststart", str(final_paths[name]),
    ], capture_output=True, text=True)
    if process.returncode:
        raise RuntimeError(f"H.264変換失敗 ({name}): {process.stderr}")

# 最終ファイルを読み戻し、フレーム数・寸法・FPSの一致を検証。
verification = {}
for name, path in final_paths.items():
    check = cv2.VideoCapture(str(path))
    count = 0
    try:
        actual_fps = check.get(cv2.CAP_PROP_FPS)
        while True:
            ok, image = check.read()
            if not ok:
                break
            if (image.shape[1], image.shape[0]) != frame_size:
                raise RuntimeError(f"出力サイズ不一致: {name}")
            count += 1
    finally:
        check.release()
    if count != written or not math.isclose(actual_fps, fps, rel_tol=0.0001):
        raise RuntimeError(f"出力検証失敗: {name}, frames={count}, fps={actual_fps}")
    verification[name] = {"frames": count, "fps": actual_fps, "file": path.name}

with (OUTPUT_DIR / "tracking.csv").open("w", newline="", encoding="utf-8") as csv_file:
    csv_writer = csv.DictWriter(csv_file, fieldnames=list(tracking_rows[0]))
    csv_writer.writeheader()
    csv_writer.writerows(tracking_rows)

manifest = {
    "tracking": {"algorithm": "DaSiamRPN", "object_count": 3,
                 "models": {k: str(v.resolve()) for k, v in TRACKING_MODELS.items()},
                 "opencv": cv2.__version__, "backend": "OpenCV CPU",
                 "initial_rois": selected_rois, "min_score": TRACKING_MIN_SCORE,
                 "coordinate_system": "after rotation/crop, before resize; xywh pixels",
                 "csv": "tracking.csv", "reference": "diabolo_tracking/dev.ipynb"},
    "input": str(INPUT_VIDEO.resolve()), "model": MODEL,
    "ultralytics": ultralytics.__version__, "device": DEVICE,
    "image_size": IMAGE_SIZE, "person_confidence": PERSON_CONFIDENCE,
    "keypoint_confidence": KEYPOINT_CONFIDENCE, "crop_xywh": CROP_XYWH,
    "person_selection": "largest_bbox_area_per_frame",
    "requested_start_seconds": START_SECONDS, "start_frame": start_frame,
    "requested_duration_seconds": DURATION_SECONDS, "frames": written,
    "fps": fps, "duration_seconds": written / fps, "size": frame_size,
    "frames_without_person": missing, "audio": False, "outputs": verification,
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
for path in raw_paths.values():
    path.unlink()  # この実行で作成し、変換・検証に成功した中間ファイルのみ削除
print("完了:", OUTPUT_DIR)
print(f"各素材 {written}フレーム / {fps:.3f}fps / {written / fps:.2f}秒 / {frame_size}")
print("人物未検出:", missing, "フレーム")


## 4. 素材を確認

`skeleton_tracking.mp4` は黒背景に骨格＋3つの追跡枠を表示します。

まず `pose_tracking.mp4` で骨格と3つの追跡枠を確認してください。
`tracking.mp4` は骨格なしで追跡枠を確認でき、`original.mp4` は同条件の元映像です。
各プレイヤーは自動同期しません。CSVの `initialized` は手動指定、`tracked` は閾値を満たす推定、`uncertain` / `invalid_box` は不確かな結果です。
交差後も同じディアボロを追えているか目視で確認してください。必要なら開始時刻と領域を指定し直します。


In [ ]:
for name in ("skeleton_tracking", "pose_tracking", "tracking", "original"):
    print(name, final_paths[name])
    display(Video(filename=str(final_paths[name]), embed=True, width=480))
print("座標:", OUTPUT_DIR / "tracking.csv")
